In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import LogisticRegression

# from sklearn.naive_bayes import MultinomialNB
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.svm import LinearSVC

In [ ]:
## Loading the dfs
df = pd.read_csv("everything.csv", index_col=0)
df_doc = pd.read_csv('specialists.csv')

In [ ]:
# Let's fetch all the possible text data
col = ['label', 'anat_symp_lem', 'symp_lem']
df = df[col]
df = df[(pd.notnull(df['anat_symp_lem'])) & (pd.notnull(df['symp_lem']))]
df['category_id'] = df['label'].factorize()[0]
category_id_df = df[['label', 'category_id']].drop_duplicates().sort_values('category_id')
category_to_id = dict(category_id_df.values)
id_to_category = dict(category_id_df[['category_id', 'label']].values)

In [ ]:
#Initialize models and tools
vectorizer = CountVectorizer()
tfidfzer = TfidfTransformer()
lr = LogisticRegression(penalty='l2', dual=False, tol=0.0001, C=1.0, fit_intercept=True, intercept_scaling=1, class_weight=None, random_state=42, solver='newton-cg', max_iter=100, multi_class='auto', verbose=0, warm_start=False, n_jobs=None, l1_ratio=None)

In [ ]:
#Split train and test data
X_train, X_test, y_train, y_test = train_test_split(df['anat_symp_lem'], df['label'], random_state=50)


#Transform X_train
vectors = vectorizer.fit_transform(X_train)
tfidf = tfidfzer.fit_transform(vectors)

#Train model
model = lr.fit(tfidf,y_train)
clf = lr.fit(X_train_tfidf, y_train)
from sklearn import metrics
vectors_test = vectorizer.transform(X_test)
pred = lr.predict(vectors_test)

acc_score = metrics.accuracy_score(y_test, pred)
f1_score = metrics.f1_score(y_test, pred, average='macro')
print("--------------------------------------------------")
print('  Accuracy: {}%'.format(round(acc_score*100,3)))
print('       F1 : {}%'.format(round(f1_score*100,3)))
print("--------------------------------------------------")

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, pred)

In [ ]:
text = ["I have a very severe and throbbing ear."]
# doc = ''.join(list(lr.predict(vectorizer.transform(text))))
result = model.predict(vectorizer.transform(text))[0]

result

In [ ]:
# # Saving model
# filename = "CheckApp_LR_Model.pickle"  

# with open(filename, 'wb') as file:  
#     pickle.dump(model, file)
    
# # Saving model
# filename = "vectorizer.pickle"  

# with open(filename, 'wb') as file:  
#     pickle.dump(vectorizer, file)

In [ ]:
# Output Results
location = "manila"
df_res = df_doc[(df_doc['Specialty ']==result) & (df_doc["Location"].str.lower().str.contains(location))]
df_online = df_doc.replace(np.nan, '', regex=True)
df_online = df_online[(df_online['Specialty ']==result) & (df_online['Online']!="")] 

In [ ]:
# Print results
if len(df_res) == 0:
    print(f"""      Searched for: {result} in {location}
      None found in the list
      
      Another option is consulting via telemedicine with the following specialists:
      """)
    for index,row in df_online.iterrows():
        print(f"""
            Doctor's Name: {row["Doctor"]}
            Link for Telemedicine Consult: {row["Online"]}
        """)
else:
    print(f"       Searched for: {result} in {location}")
    for index,row in df_res.iterrows():
        print(f"""
            Doctor's Name: {row["Doctor"]}
            Clinic Address: {(row["Clinic"])}
            Contact Details For Appointment: {row["Contact"]}
        """)

In [ ]:
df_doc